# Phase 5 — YARN + HDFS Validation

**Before running:** select the **PySpark** kernel from the kernel dropdown.

This notebook validates:
- Livy routes jobs to **YARN** (not Spark standalone)
- HDFS is reachable and `fs.defaultFS = hdfs://namenode:9000`
- Hive catalog still works with the warehouse on HDFS
- Data survives a session restart (HDFS persistence)

**UIs to keep open in parallel:**
- YARN ResourceManager: http://localhost:8088
- HDFS NameNode: http://localhost:9870

## Step 0 — Verify Livy session

In [ ]:
%%info

> **Pass**: state = `idle`. Note the session ID — it will appear in the YARN UI under Running Applications.

## Step 1 — Verify Spark master is YARN

In [ ]:
print("Spark master:", sc.master)
print("Deploy mode: ", sc._conf.get("spark.submit.deployMode"))
print("Driver host: ", sc._conf.get("spark.driver.host"))

> **Pass**:
> - `sc.master` = `yarn`
> - `deployMode` = `client`
> - `driver.host` = `livy`

## Step 2 — Verify HDFS is the default filesystem

In [ ]:
hadoopConf = sc._jsc.hadoopConfiguration()
print("fs.defaultFS:", hadoopConf.get("fs.defaultFS"))
print("auth type:   ", hadoopConf.get("hadoop.security.authentication"))

> **Pass**: `fs.defaultFS = hdfs://namenode:9000`, `auth type = simple`

## Step 3 — Check Hive catalog + warehouse location

In [ ]:
print("Catalog:",   spark.conf.get("spark.sql.catalogImplementation"))
print("Metastore:", spark.conf.get("spark.hadoop.hive.metastore.uris"))
print("Warehouse:", spark.conf.get("hive.metastore.warehouse.dir", "(from HMS)"))

> **Pass**:
> - `catalogImplementation = hive`
> - `metastore.uris = thrift://hive-metastore:9083`

## Step 4 — Create Hive database + table on HDFS

In [ ]:
%%sql
CREATE DATABASE IF NOT EXISTS risk_dw
COMMENT 'Phase 5 risk data warehouse on HDFS'

In [ ]:
%%sql
SHOW DATABASES

## Step 5 — Load data into a Hive-managed table on HDFS

In [ ]:
import random, string

traders     = ["alice", "bob", "carol", "dave"]
instruments = ["AAPL", "GOOG", "MSFT", "AMZN", "TSLA", "NVDA"]

rows = [
    (
        ''.join(random.choices(string.ascii_uppercase + string.digits, k=10)),
        random.choice(traders),
        random.choice(instruments),
        round(random.uniform(1_000, 1_000_000), 2),
        f"2026-{random.randint(1,12):02d}-{random.randint(1,28):02d}",
    )
    for _ in range(10_000)
]

from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField("trade_id",   StringType()),
    StructField("trader",     StringType()),
    StructField("instrument", StringType()),
    StructField("notional",   DoubleType()),
    StructField("trade_date", StringType()),
])

df = spark.createDataFrame(rows, schema)
df.write.mode("overwrite").saveAsTable("risk_dw.trades")

count = spark.table("risk_dw.trades").count()
print(f"Loaded {count:,} rows into risk_dw.trades")
assert count == 10_000, f"Expected 10,000 rows, got {count}"

## Step 6 — Analytics on HDFS-backed table

In [ ]:
%%sql
SELECT trader, ROUND(SUM(notional), 2) AS total_notional
FROM risk_dw.trades
GROUP BY trader
ORDER BY total_notional DESC

## Step 7 — Verify Parquet files are on HDFS

In [ ]:
# List the warehouse directory from within Spark using the Hadoop FileSystem API
fs     = sc._jvm.org.apache.hadoop.fs.FileSystem.get(sc._jsc.hadoopConfiguration())
path   = sc._jvm.org.apache.hadoop.fs.Path("/user/hive/warehouse")
files  = fs.listStatus(path)

print("Contents of hdfs:///user/hive/warehouse:")
for f in files:
    print(" ", f.getPath().getName(), "  isDir:", f.isDirectory())

> **Pass**: `risk_dw.db` directory is listed on HDFS.

You can also verify from the terminal:
```powershell
docker exec namenode hdfs dfs -ls -R /user/hive/warehouse
```

## Step 8 — Persistence test (session restart)

1. Click the SparkMagic toolbar button **Delete session** (or use `%manage_spark`).
2. A new session starts automatically on the next cell execution.
3. **Do not re-run Step 5** — the data must already be there on HDFS.

In [ ]:
count = spark.table("risk_dw.trades").count()
print(f"Rows after session restart: {count:,}")
assert count == 10_000, f"Expected 10,000 rows, got {count}"

> **Pass**: `Rows after session restart: 10,000`

> **Fail**: If count is 0, the HDFS volume was not persistent. Check that `docker compose down -v` was not run.